# DermAssist AI: Colab quickstart (Phase 2)

Sets up the repo on a fresh Colab runtime, runs the tests, validates HAM10000 from Google Drive and creates the fixed lesion-level split.

> Educational and research use only. Not a diagnostic tool.

Phase 2 runs on CPU. A GPU runtime (Runtime > Change runtime type) is only needed from Phase 3.

## 1. Check the hardware

In [ ]:
import platform
print("Python", platform.python_version())
try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch is not installed (it is pre-installed on Colab)")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Get the code

Two ways, pick one:

* **GitHub (default):** the code is cloned from `REPO_URL`. Re-running this cell pulls the latest code. If the repository is **private**, Colab needs a GitHub token: see `docs/COLAB_SETUP.md`.
* **Zip on Drive (no GitHub):** set `REPO_URL = ""` and upload `DermAssist-AI-code.zip` to the top of your Google Drive ("My Drive"). Re-running this cell unpacks the zip again.

`sh()` and `py()` run commands and **stop the notebook if a command fails**, so an error is never silently skipped.

In [ ]:
import os, subprocess, sys
from pathlib import Path

def sh(*cmd):
    """Run a shell command, show its output, and raise if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")

def py(*args):
    """Run Python (the notebook's own interpreter) with the given arguments."""
    sh(sys.executable, *args)

REPO_URL = "https://github.com/SatyaSreejay/DermAssistAI.git"  # set to "" to use the code zip instead
CODE_ZIP = Path("/content/drive/MyDrive/DermAssist-AI-code.zip")
REPO_DIR = Path("/content/DermAssist-AI")

if REPO_URL:
    if (REPO_DIR / ".git").exists():
        sh("git", "-C", REPO_DIR, "pull", "-q")
    else:
        sh("git", "clone", "-q", REPO_URL, REPO_DIR)
elif CODE_ZIP.exists():
    sh("unzip", "-q", "-o", CODE_ZIP, "-d", REPO_DIR.parent)  # the zip contains a DermAssist-AI/ folder
else:
    raise FileNotFoundError(f"REPO_URL is empty and no code zip was found at {CODE_ZIP}")
os.chdir(REPO_DIR)
py("-m", "pip", "install", "-q", "-r", "requirements-core.txt")
print("Working in", Path.cwd())

## 4. Run the tests (synthetic data only)

In [ ]:
py("-m", "pytest", "-q")

## 5. Point at the data

* `DRIVE_DATA`: the Drive folder with `HAM10000_metadata.csv` and the images (any sub-folder layout).
* `DRIVE_OUT`: Colab's disk is wiped when the session ends, so outputs that must survive (the split, reports, checkpoints) go to Drive.
* Reading ~10,000 small files straight from Drive is slow. If the zip is on Drive, set `ZIP_ON_DRIVE`; it is copied and unzipped once to the fast local disk.

In [ ]:
DRIVE_DATA = Path("/content/drive/MyDrive/HAM10000")          # TODO: adjust
DRIVE_OUT  = Path("/content/drive/MyDrive/DermAssist-AI")    # outputs that must survive the session
ZIP_ON_DRIVE = None  # e.g. DRIVE_DATA / "archive.zip"

IMAGE_DIR = DRIVE_DATA
if ZIP_ON_DRIVE:
    LOCAL = Path("/content/ham")
    if not (LOCAL / ".unzipped").exists():
        LOCAL.mkdir(parents=True, exist_ok=True)
        sh("cp", ZIP_ON_DRIVE, LOCAL / "data.zip")
        sh("unzip", "-q", "-n", LOCAL / "data.zip", "-d", LOCAL)
        (LOCAL / "data.zip").unlink()
        (LOCAL / ".unzipped").touch()
    IMAGE_DIR = LOCAL

# Use the metadata on Drive if present, otherwise look for it inside the image folder.
METADATA = DRIVE_DATA / "HAM10000_metadata.csv"
if not METADATA.exists():
    found = sorted(IMAGE_DIR.rglob("HAM10000_metadata.csv"))
    METADATA = found[0] if found else METADATA
PROCESSED, REPORTS = DRIVE_OUT / "processed", DRIVE_OUT / "reports"
if not METADATA.exists():
    raise FileNotFoundError(f"HAM10000_metadata.csv not found in {DRIVE_DATA} or {IMAGE_DIR}")
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Image folder not found: {IMAGE_DIR}")
print("metadata:", METADATA)
print("images:  ", IMAGE_DIR)

## 6. Validate metadata and images
Add `"--overwrite"` to the call only if you deliberately want to replace an earlier report.

In [ ]:
py("-m", "scripts.validate_data", "--metadata", METADATA, "--image-dir", IMAGE_DIR,
   "--processed-dir", PROCESSED, "--reports-dir", REPORTS)

## 7. Create the fixed lesion-level split
This refuses to run if `splits.csv` already exists, which protects the split that earlier experiments used.

In [ ]:
py("-m", "scripts.make_splits", "--processed-dir", PROCESSED, "--reports-dir", REPORTS)

## 8. Look at the results
Check: no `ERROR` lines above, every class in every split, and read any `WARNING` about rare classes.

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(REPORTS / "split_summary.csv", index_col=0))
display(Image(str(REPORTS / "class_distribution.png"), width=600))
display(Image(str(REPORTS / "split_proportions.png"), width=700))